# Project Group7 : Yihang Wang, Aaron Sukare

## Research Question3: What aspects of Agentic-PRs receive the most attention during review?

In [1]:
import pandas as pd
import numpy as np

In [ ]:
#load data
pr_comments_df = pd.read_parquet("hf://datasets/hao-li/AIDev/pr_comments.parquet")
pr_reviews_df = pd.read_parquet("hf://datasets/hao-li/AIDev/pr_reviews.parquet")
pr_df = pd.read_parquet("hf://datasets/hao-li/AIDev/pull_request.parquet")
print("pr_comments shape:",pr_comments_df.shape)
print("pr_reviews shape:",pr_reviews_df.shape)
print("pull_request shape:",pr_df.shape)
print("pr_comments columns sample:",list(pr_comments_df.columns)[:10])
print("pr_reviews columns sample:",list(pr_reviews_df.columns)[:10])
print("pull_request columns sample:",list(pr_df.columns)[:10])
print("agent counts sample:")
print(pr_df["agent"].value_counts().head(10))
#keep only PRs where agent is present
pr_df = pr_df[pr_df["agent"].notna()]
agentic_pr_ids = set(pr_df["id"])
print("pull_request with_agent shape:",pr_df.shape)
print("num_agentic_pr_ids:",len(agentic_pr_ids))

pr_comments shape: (39122, 7)
pr_reviews shape: (28875, 7)
pull_request shape: (33596, 14)
pr_comments columns sample: ['id', 'pr_id', 'user', 'user_id', 'user_type', 'created_at', 'body']
pr_reviews columns sample: ['id', 'pr_id', 'user', 'user_type', 'state', 'submitted_at', 'body']
pull_request columns sample: ['id', 'number', 'title', 'body', 'agent', 'user_id', 'user', 'state', 'created_at', 'closed_at']
agent counts sample:
agent
OpenAI_Codex    21799
Copilot          4970
Devin            4827
Cursor           1541
Claude_Code       459
Name: count, dtype: int64
pull_request with_agent shape: (33596, 14)
num_agentic_pr_ids: 33596


In [3]:
#build simple categories for comments and reviews
def classify_comment(text):
    if not isinstance(text,str):
        return "other"
    t = text.lower()
    if any(w in t for w in ["security","xss","injection","csrf","vulnerability","sql injection"]):
        return "security"
    if any(w in t for w in ["test","tests","testing","unit test","integration test","pytest","coverage","ci"]):
        return "testing"
    if any(w in t for w in ["style","format","formatting","lint","linter","prettier","eslint","whitespace","indent","naming"]):
        return "style"
    if any(w in t for w in ["bug","incorrect","wrong","fix","error","fails","failing","does not work","doesn't work"]):
        return "correctness"
    return "other"

need_comment_cols = ["id","pr_id","body","created_at"]
have_comment_cols = [c for c in need_comment_cols if c in pr_comments_df.columns]
pr_comments = pr_comments_df[have_comment_cols].copy()
if "created_at" in pr_comments.columns:
    pr_comments["created_at"] = pd.to_datetime(pr_comments["created_at"],errors="coerce")
pr_comments = pr_comments[pr_comments["pr_id"].isin(agentic_pr_ids)]
pr_comments["category"] = pr_comments["body"].apply(classify_comment)
print("pr_comments filtered shape:",pr_comments.shape)

need_review_cols = ["id","pr_id","body","state","submitted_at"]
have_review_cols = [c for c in need_review_cols if c in pr_reviews_df.columns]
pr_reviews = pr_reviews_df[have_review_cols].copy()
if "submitted_at" in pr_reviews.columns:
    pr_reviews["submitted_at"] = pd.to_datetime(pr_reviews["submitted_at"],errors="coerce")
pr_reviews = pr_reviews[pr_reviews["pr_id"].isin(agentic_pr_ids)]
pr_reviews["category"] = pr_reviews["body"].apply(classify_comment)
print("pr_reviews filtered shape:",pr_reviews.shape)

pr_comments filtered shape: (39122, 5)
pr_reviews filtered shape: (28875, 6)


In [4]:
#aggregate counts per category
comment_counts = pr_comments.groupby(["pr_id","category"])["id"].count().reset_index(name="num_comments")
review_counts = pr_reviews.groupby(["pr_id","category"])["id"].count().reset_index(name="num_reviews")
combined = comment_counts.merge(review_counts,on=["pr_id","category"],how="outer").fillna(0)
combined["total_mentions"] = combined["num_comments"] + combined["num_reviews"]
print("combined shape:",combined.shape)
combined.head()

combined shape: (25839, 5)


,pr_id,category,num_comments,num_reviews,total_mentions
0,2756921963,other,1.0,30.0,31.0
1,2756921963,security,1.0,0.0,1.0
2,2756921963,testing,1.0,0.0,1.0
3,2757103560,other,1.0,0.0,1.0
4,2757103560,style,1.0,0.0,1.0


In [5]:
#overall and per PR stats for categories
cat_totals = combined.groupby("category")["total_mentions"].sum().reset_index().sort_values("total_mentions",ascending=False)
print("total_mentions_by_category")
print(cat_totals)

pr_with_cat = combined[combined["total_mentions"] > 0].groupby("category")["pr_id"].nunique().reset_index(name="num_prs_with_this_category").sort_values("num_prs_with_this_category",ascending=False)
print("num_prs_with_each_category")
print(pr_with_cat)

main_cats = ["correctness","style","security","testing"]
main_cat_totals = cat_totals[cat_totals["category"].isin(main_cats)]
main_cat_prs = pr_with_cat[pr_with_cat["category"].isin(main_cats)]
print("main_categories_total_mentions")
print(main_cat_totals)
print("main_categories_pr_coverage")
print(main_cat_prs)

total_mentions_by_category
      category  total_mentions
1        other         35537.0
4      testing         25859.0
0  correctness          3041.0
2     security          2089.0
3        style          1471.0
num_prs_with_each_category
      category  num_prs_with_this_category
4      testing                       11816
1        other                        9562
0  correctness                        1915
2     security                        1557
3        style                         989
main_categories_total_mentions
      category  total_mentions
4      testing         25859.0
0  correctness          3041.0
2     security          2089.0
3        style          1471.0
main_categories_pr_coverage
      category  num_prs_with_this_category
4      testing                       11816
0  correctness                        1915
2     security                        1557
3        style                         989


In [6]:
#optional view by agent
combined_with_agent = combined.merge(pr_df[["id","agent"]],left_on="pr_id",right_on="id",how="left")
agent_cat = combined_with_agent.groupby(["agent","category"])["total_mentions"].sum().reset_index()
agent_cat = agent_cat.sort_values(["agent","total_mentions"],ascending=[True,False])
print("agent_category_pairs_sample")
print(agent_cat.head(20))

agent_category_pairs_sample
          agent     category  total_mentions
1   Claude_Code        other           682.0
4   Claude_Code      testing           547.0
2   Claude_Code     security           184.0
0   Claude_Code  correctness            72.0
3   Claude_Code        style            12.0
6       Copilot        other         20541.0
9       Copilot      testing          7034.0
5       Copilot  correctness          1787.0
8       Copilot        style           940.0
7       Copilot     security           359.0
14       Cursor      testing          1910.0
11       Cursor        other          1550.0
10       Cursor  correctness           218.0
12       Cursor     security           191.0
13       Cursor        style            58.0
19        Devin      testing         10687.0
16        Devin        other          8502.0
17        Devin     security           616.0
15        Devin  correctness           549.0
18        Devin        style           226.0
